# 🔧 02 - Preprocessing & Resampling Strategy Comparison

**Critical design decisions:**
1. Train/test split **BEFORE** any resampling (avoids data leakage)
2. StandardScaler fitted **ONLY** on training data
3. Compare 4 resampling strategies head-to-head

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from src.data_loader import load_data
from src.preprocessing import FraudPreprocessor, Resampler, get_class_weights

df, stats = load_data()
print('Dataset loaded.')
print(f'Shape: {df.shape}')
print(f'Fraud rate: {stats["fraud_rate_pct"]:.4f}%')

## 1. Train/Test Split — The Right Way

**Why split first?**
- If we SMOTE the full dataset, synthetic fraud samples will be neighbors of test data
- This leaks information about the test set into training
- Our evaluation metrics would be overly optimistic

In [ ]:
# Split FIRST, then preprocess
preprocessor = FraudPreprocessor(test_size=0.2, random_state=42)
data = preprocessor.full_preprocess(df)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

print(f'Train: {X_train.shape[0]} samples ({y_train.sum()} fraud, {y_train.mean():.4%})')
print(f'Test:  {X_test.shape[0]} samples ({y_test.sum()} fraud, {y_test.mean():.4%})')
print(f'\nFeatures: {list(X_train.columns)}')

## 2. Resampling Strategy Comparison

| Strategy | Description | Pros | Cons |
|----------|-------------|------|------|
| **None** | No resampling (use class weights) | Simple, no synthetic data | May not help all models |
| **Random Undersampling** | Remove majority samples | Fast, reduces training time | Loses information |
| **SMOTE** | Synthetic Minority Oversampling | Creates new minority samples | Can create noisy samples |
| **ADASYN** | Adaptive Synthetic Sampling | Focuses on hard-to-learn regions | More complex |
| **SMOTE+Tomek** | SMOTE with Tomek link cleanup | Cleaner boundaries | Slowest |

In [ ]:
# Compare resampling strategies
resampler = Resampler(random_state=42)
strategies = ['none', 'random_under', 'smote', 'adasyn', 'smote_tomek']
resampled_data = resampler.compare_strategies(X_train, y_train, strategies)

# Summary table
summary_rows = []
for strat, (X_r, y_r) in resampled_data.items():
    summary_rows.append({
        'Strategy': strat,
        'Total Samples': len(X_r),
        'Fraud Samples': int(y_r.sum()),
        'Fraud Rate': f'{y_r.mean():.2%}',
        'Majority:Minority': f'{(y_r == 0).sum()}:{int(y_r.sum())}',
    })

summary_df = pd.DataFrame(summary_rows)
print('\n=== Resampling Strategy Comparison ===')
print(summary_df.to_string(index=False))

In [ ]:
# Visualize class distribution for each strategy
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for idx, (strat, (X_r, y_r)) in enumerate(resampled_data.items()):
    ax = axes[idx]
    counts = y_r.value_counts()
    bars = ax.bar(['Legit', 'Fraud'], [counts.get(0, 0), counts.get(1, 0)],
                  color=['#2ecc71', '#e74c3c'], edgecolor='white')
    ax.set_title(strat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(counts.values) * 1.1)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 100,
                f'{val:,}', ha='center', fontsize=9)

plt.suptitle('Class Distribution by Resampling Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/processed/resampling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Class Weights

An alternative to resampling: tell the model to penalize misclassifying the minority class more heavily.

In [ ]:
class_weights = get_class_weights(y_train)
print(f'\nClass weights:')
print(f'  Legitimate (0): {class_weights[0]:.2f}')
print(f'  Fraud (1):      {class_weights[1]:.2f}')
print(f'\n  Fraud class is weighted {class_weights[1]/class_weights[0]:.0f}x more than legitimate.')

## 4. Save Preprocessed Data

Save the preprocessed splits for use in the modeling notebook.

In [ ]:
# Save preprocessed data
import joblib

os.makedirs('data/processed', exist_ok=True)

joblib.dump(X_train, 'data/processed/X_train.pkl')
joblib.dump(X_test, 'data/processed/X_test.pkl')
joblib.dump(y_train, 'data/processed/y_train.pkl')
joblib.dump(y_test, 'data/processed/y_test.pkl')
preprocessor.save_scaler('models/scaler.pkl')

# Also save resampled versions for comparison
for strat, (X_r, y_r) in resampled_data.items():
    joblib.dump(X_r, f'data/processed/X_train_{strat}.pkl')
    joblib.dump(y_r, f'data/processed/y_train_{strat}.pkl')

print('\nAll preprocessed data saved.')
print('\n✅ Preprocessing complete. Ready for modeling.')